# Dashboard HBO Max · Películas y Series
Dashboard interactivo (Dash + Plotly) sobre el catálogo de HBO Max.
El código está separado en tres capas: **datos** (`data`: lectura y limpieza), **lógica** (`logic`: filtros y métricas) y **vistas** (`views`: layout, figuras y callbacks).

## 1. Dependencias

In [ ]:
!pip install -q dash plotly pandas

## 2. Datos
Sube `titles.csv` y `credits.csv` (Archivos de Colab) o colócalos en la carpeta `archive/`.

In [ ]:
from pathlib import Path

DATA_DIR = next((p for p in (Path("archive"), Path("/content")) if (p / "titles.csv").exists()), None)
if DATA_DIR is None:
    from google.colab import files
    files.upload()
    DATA_DIR = Path(".")
print("Datos en:", DATA_DIR.resolve())

## 3. Módulos
Cada celda escribe un módulo del paquete `hbo_max_dashboard` (capas `data`, `logic` y `views`).

In [ ]:
from pathlib import Path

for sub in ('', 'data', 'logic', 'views'):
    Path('hbo_max_dashboard', sub).mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile hbo_max_dashboard/__init__.py
"""Dashboard interactivo del catálogo de HBO Max.

Capas (la dependencia solo apunta hacia abajo):
    views  ->  logic      (Dash/Plotly; filtros y métricas)
    app    ->  data       (composición; lectura y limpieza de CSV)
"""

In [ ]:
%%writefile hbo_max_dashboard/config.py
"""Constantes compartidas: rutas, etiquetas y límites."""
from pathlib import Path

DATA_DIR = Path(__file__).resolve().parents[2] / "archive"

TYPE_LABELS = {"MOVIE": "Película", "SHOW": "Serie"}
ALL_GENRES = "__all__"

TOP_N = 10
MIN_VOTES = 1000  # votos mínimos para promediar IMDb por género

In [ ]:
%%writefile hbo_max_dashboard/data/__init__.py
"""Capa de datos: lectura y limpieza. No conoce la lógica de negocio ni la UI."""
from .repository import load_credits, load_titles

__all__ = ["load_titles", "load_credits"]

In [ ]:
%%writefile hbo_max_dashboard/data/cleaning.py
"""Limpieza y normalización de los datos crudos."""
import ast

import pandas as pd

from ..config import TYPE_LABELS


def _to_list(text) -> list:
    """Convierte "['drama', 'crime']" en ['drama', 'crime']."""
    try:
        return ast.literal_eval(text) if isinstance(text, str) else []
    except (ValueError, SyntaxError):
        return []


def clean_titles(raw: pd.DataFrame) -> pd.DataFrame:
    titles = raw.copy()
    titles["genres"] = titles["genres"].apply(_to_list)
    titles["production_countries"] = titles["production_countries"].apply(_to_list)
    titles["tipo"] = titles["type"].map(TYPE_LABELS)
    titles["age_certification"] = titles["age_certification"].fillna("Sin dato")
    return titles

In [ ]:
%%writefile hbo_max_dashboard/data/repository.py
"""Acceso a las fuentes de datos (CSV). Único módulo que toca el disco."""
from pathlib import Path

import pandas as pd

from ..config import DATA_DIR
from .cleaning import clean_titles


def read_csv(name: str, data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Lee `name.csv` o `name_csv.zip`, lo que exista."""
    for candidate in (f"{name}.csv", f"{name}_csv.zip"):
        path = Path(data_dir) / candidate
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(f"No se encontró {name}.csv ni {name}_csv.zip en {data_dir}")


def load_titles(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    return clean_titles(read_csv("titles", data_dir))


def load_credits(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    return read_csv("credits", data_dir)

In [ ]:
%%writefile hbo_max_dashboard/logic/__init__.py
"""Capa de lógica: filtros y métricas puras sobre DataFrames. No importa Dash ni Plotly."""

In [ ]:
%%writefile hbo_max_dashboard/logic/filters.py
"""Reglas de selección: qué títulos entran según los filtros del usuario."""
import pandas as pd

from ..config import ALL_GENRES


def filter_titles(titles: pd.DataFrame, types: list[str], genre: str, years: tuple[int, int]) -> pd.DataFrame:
    mask = titles["tipo"].isin(types) & titles["release_year"].between(*years)
    if genre != ALL_GENRES:
        mask &= titles["genres"].apply(lambda g: genre in g)
    return titles[mask]


def genre_options(titles: pd.DataFrame) -> list[str]:
    return sorted(titles["genres"].explode().dropna().unique())


def year_bounds(titles: pd.DataFrame) -> tuple[int, int]:
    return int(titles["release_year"].min()), int(titles["release_year"].max())

In [ ]:
%%writefile hbo_max_dashboard/logic/metrics.py
"""Indicadores y agregaciones sobre una selección de títulos."""
import pandas as pd

from ..config import MIN_VOTES, TOP_N


def kpis(df: pd.DataFrame) -> dict:
    return {
        "total": len(df),
        "movies": int((df["tipo"] == "Película").sum()),
        "shows": int((df["tipo"] == "Serie").sum()),
        "imdb_mean": df["imdb_score"].mean(),
    }


def titles_per_year(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby(["release_year", "tipo"]).size().reset_index(name="titulos")


def top_genres(df: pd.DataFrame, n: int = TOP_N) -> pd.DataFrame:
    counts = df["genres"].explode().dropna().value_counts().head(n)
    return counts.rename_axis("genero").reset_index(name="titulos")


def top_countries(df: pd.DataFrame, n: int = TOP_N) -> pd.DataFrame:
    counts = df["production_countries"].explode().dropna().value_counts().head(n)
    return counts.rename_axis("pais").reset_index(name="titulos")


def top_people(df: pd.DataFrame, credits: pd.DataFrame, role: str, n: int = TOP_N) -> pd.DataFrame:
    """Personas con más títulos dentro de la selección actual."""
    sel = credits[(credits["role"] == role) & credits["id"].isin(df["id"])]
    counts = sel.groupby("name")["id"].nunique().nlargest(n)
    return counts.rename("titulos").reset_index()


def certification_counts(df: pd.DataFrame) -> pd.DataFrame:
    counts = df["age_certification"].value_counts()
    return counts.rename_axis("clasificacion").reset_index(name="titulos")


def imdb_by_genre(df: pd.DataFrame, n: int = TOP_N, min_votes: int = MIN_VOTES) -> pd.DataFrame:
    """IMDb promedio por género, ignorando títulos con pocos votos."""
    rated = df[df["imdb_votes"] >= min_votes].explode("genres").dropna(subset=["genres", "imdb_score"])
    stats = rated.groupby("genres")["imdb_score"].agg(["mean", "size"])
    stats = stats[stats["size"] >= 10].nlargest(n, "mean")
    return stats["mean"].rename("imdb").rename_axis("genero").reset_index()

In [ ]:
%%writefile hbo_max_dashboard/views/__init__.py
"""Capa de vistas: layout, figuras y callbacks de Dash. Consume la capa de lógica."""

In [ ]:
%%writefile hbo_max_dashboard/views/theme.py
"""Paleta y estilos visuales compartidos por las vistas."""
PRIMARY = "#9B5CF6"
PALETTE = {"Película": PRIMARY, "Serie": "#2DD4BF"}
BG = "#10091c"
CARD = "#1b1230"
TEXT = "#f2f0f7"
MUTED = "#9d93b5"

In [ ]:
%%writefile hbo_max_dashboard/views/figures.py
"""Construcción de figuras Plotly (vista). Cada función recibe datos ya agregados."""
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from ..logic import metrics
from .theme import BG, CARD, MUTED, PALETTE, PRIMARY, TEXT


def _style(fig: go.Figure, title: str) -> go.Figure:
    fig.update_layout(
        title=dict(text=title, font=dict(size=14, color=TEXT), x=0.02),
        paper_bgcolor=CARD,
        plot_bgcolor=CARD,
        font=dict(color=TEXT),
        margin=dict(l=10, r=10, t=45, b=10),
        legend=dict(title=None, orientation="h", y=1.02, yanchor="bottom", x=1, xanchor="right"),
        height=340,
    )
    fig.update_xaxes(gridcolor="#333", zeroline=False)
    fig.update_yaxes(gridcolor="#333", zeroline=False)
    return fig


def empty_figure(title: str) -> go.Figure:
    fig = go.Figure()
    fig.add_annotation(text="Sin datos para esta selección", showarrow=False, font=dict(color=MUTED, size=14))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return _style(fig, title)


def _bar_h(data: pd.DataFrame, y: str, x: str, title: str, xlabel: str) -> go.Figure:
    if data.empty:
        return empty_figure(title)
    fig = px.bar(data.iloc[::-1], x=x, y=y, orientation="h", color_discrete_sequence=[PRIMARY])
    fig.update_layout(xaxis_title=xlabel, yaxis_title=None)
    return _style(fig, title)


def year_histogram(df: pd.DataFrame) -> go.Figure:
    title = "Títulos por año de estreno"
    data = metrics.titles_per_year(df)
    if data.empty:
        return empty_figure(title)
    fig = px.bar(data, x="release_year", y="titulos", color="tipo", color_discrete_map=PALETTE)
    fig.update_layout(barmode="stack", xaxis_title="Año", yaxis_title="Títulos")
    return _style(fig, title)


def genres_bar(df: pd.DataFrame) -> go.Figure:
    return _bar_h(metrics.top_genres(df), "genero", "titulos", "Géneros más frecuentes", "Títulos")


def runtime_scatter(df: pd.DataFrame) -> go.Figure:
    title = "Duración vs. puntuación IMDb"
    data = df.dropna(subset=["imdb_score", "runtime"])
    if data.empty:
        return empty_figure(title)
    fig = px.scatter(
        data, x="runtime", y="imdb_score", color="tipo", color_discrete_map=PALETTE,
        hover_name="title", hover_data={"release_year": True, "tipo": False}, opacity=0.6,
    )
    fig.update_layout(xaxis_title="Duración (min)", yaxis_title="IMDb")
    return _style(fig, title)


def countries_bar(df: pd.DataFrame) -> go.Figure:
    data = metrics.top_countries(df)
    title = "Top 10 países productores"
    if data.empty:
        return empty_figure(title)
    fig = px.bar(data, x="pais", y="titulos", color_discrete_sequence=[PRIMARY])
    fig.update_layout(xaxis_title="País", yaxis_title="Títulos")
    return _style(fig, title)


def people_bar(df: pd.DataFrame, credits: pd.DataFrame, role: str, title: str) -> go.Figure:
    return _bar_h(metrics.top_people(df, credits, role), "name", "titulos", title, "Títulos")


def certification_donut(df: pd.DataFrame) -> go.Figure:
    title = "Clasificación por edades"
    data = metrics.certification_counts(df)
    if data.empty:
        return empty_figure(title)
    fig = px.pie(data, names="clasificacion", values="titulos", hole=0.5,
                 color_discrete_sequence=px.colors.sequential.Purples_r)
    fig.update_traces(textinfo="percent", marker=dict(line=dict(color=BG, width=1)))
    return _style(fig, title)


def imdb_genre_bar(df: pd.DataFrame) -> go.Figure:
    data = metrics.imdb_by_genre(df)
    fig = _bar_h(data, "genero", "imdb", "IMDb promedio por género (≥1000 votos)", "IMDb promedio")
    if not data.empty:
        fig.update_xaxes(range=[max(0, data["imdb"].min() - 0.5), data["imdb"].max() + 0.2])
    return fig

In [ ]:
%%writefile hbo_max_dashboard/views/layout.py
"""Estructura HTML del dashboard (vista). No contiene lógica de datos."""
from dash import dcc, html

from ..config import ALL_GENRES, TYPE_LABELS
from .theme import BG, CARD, MUTED, PRIMARY, TEXT

CARD_STYLE = {"background": CARD, "borderRadius": "8px", "padding": "12px 16px"}
GRID_4 = {"display": "grid", "gridTemplateColumns": "repeat(auto-fit, minmax(300px, 1fr))", "gap": "12px", "marginBottom": "12px"}

CHARTS = [
    ("g-year", "Títulos por año"),
    ("g-genres", "Géneros"),
    ("g-scatter", "Duración vs IMDb"),
    ("g-countries", "Países"),
    ("g-actors", "Actores"),
    ("g-directors", "Directores"),
    ("g-cert", "Clasificación"),
    ("g-imdb-genre", "IMDb por género"),
]


def _filter(label: str, control) -> html.Div:
    return html.Div([html.Div(label, style={"marginBottom": "8px", "fontWeight": 600}), control], style=CARD_STYLE)


def _kpi(label: str, value_id: str) -> html.Div:
    return html.Div(
        [html.Div(label, style={"color": MUTED, "fontSize": "13px"}),
         html.Div(id=value_id, style={"color": PRIMARY, "fontSize": "28px", "fontWeight": 700})],
        style={**CARD_STYLE, "textAlign": "center"},
    )


def build_layout(genres: list[str], year_bounds: tuple[int, int]) -> html.Div:
    year_min, year_max = year_bounds
    return html.Div(
        style={"background": BG, "color": TEXT, "minHeight": "100vh", "padding": "20px", "fontFamily": "system-ui, sans-serif"},
        children=[
            html.H1("HBO MAX · Películas y Series", style={"color": PRIMARY, "margin": 0}),
            html.P("Explora el catálogo de HBO Max: géneros, años, puntuaciones de IMDb y elenco.", style={"color": MUTED}),
            html.Div(style={**GRID_4, "gridTemplateColumns": "1fr 1.5fr 2fr"}, children=[
                _filter("Tipo", dcc.Checklist(
                    id="f-type", value=list(TYPE_LABELS.values()), inline=True,
                    options=list(TYPE_LABELS.values()), inputStyle={"marginRight": "4px", "marginLeft": "12px"})),
                _filter("Género", dcc.Dropdown(
                    id="f-genre", value=ALL_GENRES, clearable=False,
                    options=[{"label": "Todos los géneros", "value": ALL_GENRES}] + [{"label": g, "value": g} for g in genres],
                    style={"color": "#000"})),
                _filter("Año de estreno", dcc.RangeSlider(
                    id="f-years", min=year_min, max=year_max, step=1, value=[year_min, year_max],
                    marks={y: str(y) for y in range(year_min - year_min % 10 + 10, year_max + 1, 10)},
                    tooltip={"placement": "bottom", "always_visible": False})),
            ]),
            html.Div(style=GRID_4, children=[
                _kpi("Títulos", "k-total"), _kpi("Películas", "k-movies"),
                _kpi("Series", "k-shows"), _kpi("IMDb promedio", "k-imdb"),
            ]),
            html.Div(style=GRID_4, children=[html.Div(dcc.Graph(id=cid), style=CARD_STYLE) for cid, _ in CHARTS]),
        ],
    )

In [ ]:
%%writefile hbo_max_dashboard/views/callbacks.py
"""Controlador: conecta los filtros con métricas y figuras."""
import pandas as pd
from dash import Dash, Input, Output

from ..logic import filters, metrics
from . import figures
from .layout import CHARTS

FILTERS = [Input("f-type", "value"), Input("f-genre", "value"), Input("f-years", "value")]
KPI_IDS = ["k-total", "k-movies", "k-shows", "k-imdb"]


def register_callbacks(app: Dash, titles: pd.DataFrame, credits: pd.DataFrame) -> None:
    @app.callback([Output(k, "children") for k in KPI_IDS] + [Output(cid, "figure") for cid, _ in CHARTS], FILTERS)
    def update(types, genre, years):
        df = filters.filter_titles(titles, types or [], genre, tuple(years))
        k = metrics.kpis(df)
        imdb = "–" if pd.isna(k["imdb_mean"]) else f"{k['imdb_mean']:.2f}"
        return (
            f"{k['total']:,}", f"{k['movies']:,}", f"{k['shows']:,}", imdb,
            figures.year_histogram(df),
            figures.genres_bar(df),
            figures.runtime_scatter(df),
            figures.countries_bar(df),
            figures.people_bar(df, credits, "ACTOR", "Actores con más títulos"),
            figures.people_bar(df, credits, "DIRECTOR", "Directores con más títulos"),
            figures.certification_donut(df),
            figures.imdb_genre_bar(df),
        )

In [ ]:
%%writefile hbo_max_dashboard/app.py
"""Fábrica de la aplicación Dash."""
from pathlib import Path

from dash import Dash

from .config import DATA_DIR
from .data import load_credits, load_titles
from .logic.filters import genre_options, year_bounds
from .views.callbacks import register_callbacks
from .views.layout import build_layout


def create_app(data_dir: Path = DATA_DIR) -> Dash:
    titles = load_titles(data_dir)
    credits = load_credits(data_dir)
    app = Dash(__name__, title="HBO Max · Películas y Series")
    app.layout = build_layout(genre_options(titles), year_bounds(titles))
    register_callbacks(app, titles, credits)
    return app

## 4. Ejecutar el dashboard

In [ ]:
from hbo_max_dashboard.app import create_app

app = create_app(DATA_DIR)
app.run(jupyter_mode="inline", height=1400, debug=False)